## Instructions:

- Put the parts of your code under the corresponding sections. (0.25/2 points will be taken off for not doing this.)
- Do not include any redundant/irrelevant code, text or comments. (0.5/2 points will be taken off for not doing this.)
- **Your code must run without any errors or runtime issues.** (Failure to meet this condition will result in a 0.)
- **Your code must return your Public Leaderboard score.** (Failure to meet this condition will result in a 0.)
- **Submit both your ipynb and your html file for grading purposes.**

## 1) Libraries

Put all the Python libraries and tools you imported here.

In [22]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

import os
os.environ["OMP_NUM_THREADS"] = "1"
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

## 2) Data

- This section is required to include the code that reads, cleans and preprocesses the datasets.
- Note that both the training and test datasets should undergo the same sequence of operations.

In [7]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [8]:
train['host_since'] = pd.to_datetime(train['host_since'], errors='coerce')
train['host_location'] = train['host_location'].notna().astype(int)
train['host_response_time'] = train['host_response_time'].notna().astype(int)
train['host_response_rate'] = train['host_response_rate'].str.rstrip('%').astype(float)
train['host_acceptance_rate'] = train['host_acceptance_rate'].str.rstrip('%').astype(float)
train['price'] = train['price'].replace('[\$,]', '', regex=True).astype(float)
train['first_review'] = pd.to_datetime(train['first_review'], errors='coerce')
train['last_review'] = pd.to_datetime(train['last_review'], errors='coerce')
test['host_since'] = pd.to_datetime(test['host_since'], errors='coerce')
test['host_location'] = test['host_location'].notna().astype(int)
test['host_response_time'] = test['host_response_time'].notna().astype(int)
test['host_response_rate'] = test['host_response_rate'].str.rstrip('%').astype(float)
test['host_acceptance_rate'] = test['host_acceptance_rate'].str.rstrip('%').astype(float)
test['first_review'] = pd.to_datetime(test['first_review'], errors='coerce')
test['last_review'] = pd.to_datetime(test['last_review'], errors='coerce')

In [9]:
train = pd.get_dummies(train, columns=['listing_location'])
test = pd.get_dummies(test, columns=['listing_location'])
train = pd.get_dummies(train, columns=['room_type'])
test = pd.get_dummies(test, columns=['room_type'])

In [10]:
bool_cols = train.select_dtypes(include='bool').columns
train[bool_cols] = train[bool_cols].astype(int)

bool_cols = test.select_dtypes(include='bool').columns
test[bool_cols] = test[bool_cols].astype(int)

In [11]:
X_train = train.drop(columns=['price', 'description', 'host_since', 'host_about', 'latitude', 'longitude', 'first_review', 'last_review', 'has_availability', 'amenities', 'host_location', 'host_neighbourhood'], axis=1)
y_train = train['price']
X_test = test.drop(columns=['description', 'host_since', 'host_about', 'latitude', 'longitude', 'first_review', 'last_review', 'has_availability', 'amenities', 'host_location', 'host_neighbourhood'], axis=1)

In [12]:
X_train['bathrooms'] = X_train['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)
X_test['bathrooms'] = X_test['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)
X_train = X_train.drop(columns='bathrooms_text')
X_test = X_test.drop(columns='bathrooms_text')

In [13]:
object_cols = X_train.select_dtypes(include='object').columns
X_train = pd.get_dummies(X_train, columns=object_cols, drop_first=True)

object_cols = X_test.select_dtypes(include='object').columns
X_test = pd.get_dummies(X_test, columns=object_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [14]:
for col in X_train.columns:
    if X_train[col].isnull().any():
        if X_train[col].dtype in ['float64', 'int64']:
            X_train[col] = X_train[col].fillna(X_train[col].median())
        else:
            X_train[col] = X_train[col].fillna(X_train[col].mode()[0])

for col in X_test.columns:
    if X_test[col].isnull().any():
        if X_test[col].dtype in ['float64', 'int64']:
            X_test[col] = X_test[col].fillna(X_test[col].median())
        else:
            X_test[col] = X_test[col].fillna(X_test[col].mode()[0])

In [15]:
y_train_log = np.log(y_train)

In [16]:
X_train.columns = X_train.columns.str.replace(r"[\[\]<>]", "", regex=True)
X_test.columns = X_test.columns.str.replace(r"[\[\]<>]", "", regex=True)

## 3) Machine Learning Model

- This section is required to train the **already tuned** model and obtain the test predictions (or prediction probabilities) with it.
- As written in the instructions, your code must not have any runtime issues, so **do NOT include your grid search here!** You will still need to tune your model to pass the thresholds. However, you need to keep that as your personal work and should NOT include the grid search here.

In [23]:
final_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=12,
    colsample_bytree=0.5,
    gamma=0,
    learning_rate=0.1,
    max_depth=8,
    n_estimators=400,
    reg_lambda=1,
    subsample=1
)

final_model.fit(X_train, y_train_log)

y_pred_log = final_model.predict(X_train)
y_pred = np.exp(y_pred_log)

mae_original_scale = mean_absolute_error(y_train, y_pred)
print("MAE in original scale:", mae_original_scale)

y_test_pred_log = final_model.predict(X_test)
y_test_pred = np.exp(y_test_pred_log) 

MAE in original scale: 28.839298019551066


In [2]:
# Note: On Quest, this model gave a test MAE of 96.97 but on my local laptop, it gave 99.48.
# (The training MAE on Quest was 28.839 and on my laptop it is 28.5255) due to differences in scikit-learn version or environment and 
    # the random_state being different in the two environments.

## 4) Exporting the Predictions

Include the code that (1) puts the predictions in the format that Kaggle understands and (2) exports it as a csv file.

In [24]:
submission_df = pd.DataFrame({
    'id': test['id'], 
    'predicted': np.round(y_test_pred)  
})

submission_df.to_csv('reg5.csv', index=False) 